# Patcher Training Colab (Tree-sitter + GraphRAG Dataset)

This notebook trains the patcher directly from `patcher_train/eval/test.jsonl` built by `build_patcher_data.py`.

Design goals:
- Use new structured context (planner + Tree-sitter spans + GraphRAG retrieval).
- Prioritize correctness over pure format.
- Keep robust checkpoint selection by generated patch quality.


In [ ]:
!pip -q install -U transformers datasets accelerate peft bitsandbytes sentencepiece tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import re
from dataclasses import dataclass
from typing import Dict, List, Any
from pathlib import Path
import numpy as np
import torch
from tqdm import tqdm


Mounted at /content/drive


In [ ]:
# ===== Paths / Run Config =====
# v3 dataset bundle from patch_patcher_v3/outputs
DATA_DIR = '/content/drive/MyDrive/patch_patcher_v3/outputs'
TRAIN_JSONL = f'{DATA_DIR}/patcher_train.jsonl'
EVAL_JSONL = f'{DATA_DIR}/patcher_eval.jsonl'
TEST_JSONL = f'{DATA_DIR}/patcher_test.jsonl'
REPORT_JSON = f'{DATA_DIR}/dataset_report.json'

# Overnight final run folder
ROOT_DIR = '/content/drive/MyDrive/patcher_final_overnight_run'
OUT_DIR = f'{ROOT_DIR}/run'
EXPORT_DIR = f'{ROOT_DIR}/exports'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

MODEL_NAME = 'Qwen/Qwen2.5-Coder-7B-Instruct'
SEED = 42
MAX_SEQ_LEN = 4096

# Repo clone path used by apply-and-parse eval checks.
# Make sure this repo exists in Colab and contains the same history as base_sha references.
REPO_DIR = '/content/airflow'

# Dataset shaping
KEEP_ONLY_STRUCTURAL_PASS = True
KEEP_ONLY_ALLOWED_FILES_PASS = True
KEEP_ONLY_VALID_DIFF = True

# Stage-A style precision filters
REQUIRE_TREE_SITTER_SPANS = True
REQUIRE_TOUCHED_SPAN = True
MAX_FILES_TOUCHED_FILTER = 2

# Context budget/noise controls
MAX_SPANS_PER_EXAMPLE = 8
MAX_GRAPHRAG_CANDIDATES = 3
MAX_GRAPHRAG_IDIOMS = 4
MAX_GRAPHRAG_SNIPPET_CHARS = 120

# Training
NUM_EPOCHS = 2
LR = 2e-5
GRAD_ACCUM = 8
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1

# Generation (eval, N-best, smoke) — multi-file / long hunks need headroom
PATCHER_MAX_NEW_TOKENS = 1024

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
if os.path.exists(REPORT_JSON):
    with open(REPORT_JSON, 'r', encoding='utf-8') as f:
        print(json.dumps(json.load(f), indent=2)[:4000])
else:
    print('dataset_report.json not found (optional).')


{
  "total_input_rows": 23607,
  "total_kept": 11151,
  "total_dropped": 13487,
  "dropped_by_reason": {
    "over_max_sequence_budget": 12456,
    "treesitter_blob_lookup_failures": 1031
  },
  "split_sizes": {
    "train": 8920,
    "eval": 1116,
    "test": 1115
  },
  "distributions": {
    "files_touched": {
      "count": 11151,
      "min": 1,
      "p50": 1,
      "p90": 2,
      "max": 3
    },
    "additions": {
      "count": 11151,
      "min": 0,
      "p50": 3,
      "p90": 31,
      "max": 294
    },
    "patch_tokens": {
      "count": 11151,
      "min": 8,
      "p50": 80,
      "p90": 346,
      "max": 1193
    }
  },
  "percent_single_file_rows": 76.10079813469643,
  "percent_passing_quality_labels": {
    "valid_diff": 100.0,
    "touches_only_allowed_files": 100.0,
    "compile_pass": 95.59680746121424,
    "structural_pass": 95.59680746121424
  },
  "query_reproducibility": {
    "graphrag_enabled": true,
    "graphrag_query_version": "v1_file_neighbors",
    "gr

In [ ]:
def load_jsonl(path: str) -> List[dict]:
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_JSONL)
eval_rows = load_jsonl(EVAL_JSONL)
test_rows = load_jsonl(TEST_JSONL) if os.path.exists(TEST_JSONL) else []

print('raw train/eval/test:', len(train_rows), len(eval_rows), len(test_rows))


raw train/eval/test: 8920 1116 1115


In [ ]:
def row_passes_quality(row: dict) -> bool:
    q = row.get('meta', {}).get('quality_labels', {})
    if KEEP_ONLY_VALID_DIFF and not bool(q.get('valid_diff', False)):
        return False
    if KEEP_ONLY_ALLOWED_FILES_PASS and not bool(q.get('touches_only_allowed_files', False)):
        return False
    if KEEP_ONLY_STRUCTURAL_PASS and not bool(q.get('structural_pass', False)):
        return False

    inp = row.get('input', {})
    allowed_files = (inp.get('constraints', {}) or {}).get('allowed_files', []) or []
    if len(allowed_files) > MAX_FILES_TOUCHED_FILTER:
        return False

    spans = (inp.get('treesitter_context', {}) or {}).get('spans', []) or []
    if REQUIRE_TREE_SITTER_SPANS and len(spans) == 0:
        return False
    if REQUIRE_TOUCHED_SPAN and spans and not any(bool(s.get('touched_by_gold_patch', False)) for s in spans):
        return False

    return True

train_rows = [r for r in train_rows if row_passes_quality(r)]
eval_rows = [r for r in eval_rows if row_passes_quality(r)]
if test_rows:
    test_rows = [r for r in test_rows if row_passes_quality(r)]

print('filtered train/eval/test:', len(train_rows), len(eval_rows), len(test_rows))


filtered train/eval/test: 1382 158 163


In [ ]:
def clip_text(x: str, n: int) -> str:
    x = '' if x is None else str(x)
    x = x.replace('\r', '\n').strip()
    return x[:n]

def format_planner(pl: dict) -> str:
    files = pl.get('target_files', []) or []
    funcs = pl.get('target_functions', []) or []
    lines = [
        '--- PLANNER DIRECTIVE ---',
        f"REQUIRES_CODE_CHANGE: {pl.get('requires_code_change', 'YES')}",
        f"CONFIDENCE: {pl.get('confidence', 'MEDIUM')}",
        f"REASON: {clip_text(pl.get('reason', ''), 260)}",
        'PLAN:',
        f"- Root cause: {clip_text(pl.get('root_cause', ''), 260)}",
        '- Target files:'
    ]
    lines.extend([f'  - {f}' for f in files])
    if funcs:
        lines.append('- Target functions:')
        lines.extend([f'  - {fn}' for fn in funcs[:16]])
    lines.append(f"- Test strategy: {clip_text(pl.get('test_strategy', ''), 220)}")
    return '\n'.join(lines)

def format_treesitter(ts_ctx: dict) -> str:
    spans = ts_ctx.get('spans', []) or []
    # prioritize touched_by_gold_patch spans first
    spans = sorted(spans, key=lambda s: (not bool(s.get('touched_by_gold_patch', False)), s.get('start_line', 0)))
    spans = spans[:MAX_SPANS_PER_EXAMPLE]
    out = ['--- TREE_SITTER_SPANS ---']
    if not spans:
        out.append('None')
        return '\n'.join(out)
    for s in spans:
        out.append(f"# FILE: {s.get('file', '')}")
        out.append(f"# SYMBOL: {s.get('symbol', '')} ({s.get('symbol_type', '')})")
        out.append(f"# RANGE: {s.get('start_line', 0)}-{s.get('end_line', 0)}")
        out.append(clip_text(s.get('code', ''), 2200))
        out.append('')
    return '\n'.join(out).strip()

def format_graphrag(gr: dict) -> str:
    out = ['--- HISTORICAL REPO IDIOMS (GraphRAG) ---']
    cands = (gr.get('candidate_files_topk', []) or [])[:MAX_GRAPHRAG_CANDIDATES]
    idioms = (gr.get('historical_idioms', []) or [])[:MAX_GRAPHRAG_IDIOMS]
    if cands:
        out.append('Top candidate files (high confidence only):')
        for c in cands:
            out.append(f"- {c.get('file', '')} (score={c.get('score', 0)})")
    else:
        out.append('Top candidate files: None')
    if idioms:
        out.append('Historical snippets (concise):')
        for it in idioms:
            out.append(
                f"- [{it.get('file', '')}] PR#{it.get('source_pr', '')}: "
                f"{clip_text(it.get('snippet', ''), MAX_GRAPHRAG_SNIPPET_CHARS)}"
            )
    else:
        out.append('Historical snippets: None')
    return '\n'.join(out)

def format_constraints(c: dict) -> str:
    allowed = c.get('allowed_files', []) or []
    lines = [
        '--- CONSTRAINTS ---',
        'Allowed files:'
    ]
    lines.extend([f'  - {x}' for x in allowed])
    lines.append(f"forbid_new_files: {bool(c.get('forbid_new_files', True))}")
    lines.append(f"forbid_unrelated_refactors: {bool(c.get('forbid_unrelated_refactors', True))}")
    lines.append(f"output_format: {c.get('output_format', 'unified_diff_only')}")
    return '\n'.join(lines)

def build_prompt(row: dict) -> str:
    inp = row.get('input', {})
    planner = format_planner(inp.get('planner_directive', {}))
    ts = format_treesitter(inp.get('treesitter_context', {}))
    gr = format_graphrag(inp.get('graphrag_context', {}))
    cons = format_constraints(inp.get('constraints', {}))
    instr = clip_text(inp.get('instruction', ''), 400)
    return (
        'You are an expert patch generation model.\n'
        'Generate the minimal correct unified diff patch.\n'
        'Hard rules:\n'
        '- Touch only allowed files.\n'
        '- Prefer edits inside or adjacent to touched Tree-sitter spans.\n'
        '- Keep the patch minimal and avoid unrelated refactors/reformatting.\n'
        '- If uncertain, make the smallest safe change only.\n'
        '- Output ONLY unified diff text starting with --- a/.\n\n'
        f'INSTRUCTION: {instr}\n\n'
        f'{planner}\n\n'
        f'{cons}\n\n'
        f'{gr}\n\n'
        f'{ts}\n\n'
        'TASK: Produce the patch now.\n'
    )

def build_target(row: dict) -> str:
    return row.get('output', {}).get('unified_diff', '').rstrip() + '\n'


In [ ]:
train_examples = [{
    'input': build_prompt(r),
    'output': build_target(r),
    'meta': {
        'id': r.get('id'),
        'allowed_files': r.get('input', {}).get('constraints', {}).get('allowed_files', []) or [],
        'gold_hunk_headers': r.get('output', {}).get('gold_hunk_headers', []) or [],
    }
} for r in train_rows]

eval_examples = [{
    'input': build_prompt(r),
    'output': build_target(r),
    'meta': {
        'id': r.get('id'),
        'base_sha': r.get('meta', {}).get('base_sha', ''),
        'allowed_files': r.get('input', {}).get('constraints', {}).get('allowed_files', []) or [],
        'gold_hunk_headers': r.get('output', {}).get('gold_hunk_headers', []) or [],
    }
} for r in eval_rows]

print('prepared train/eval examples:', len(train_examples), len(eval_examples))
print('\nPROMPT PREVIEW:\n')
print(train_examples[0]['input'][:3000])
print('\nTARGET PREVIEW:\n')
print(train_examples[0]['output'][:1200])


prepared train/eval examples: 1382 158

PROMPT PREVIEW:

You are an expert patch generation model.
Generate the minimal correct unified diff patch.
Hard rules:
- Touch only allowed files.
- Prefer edits inside or adjacent to touched Tree-sitter spans.
- Keep the patch minimal and avoid unrelated refactors/reformatting.
- If uncertain, make the smallest safe change only.
- Output ONLY unified diff text starting with --- a/.

INSTRUCTION: Generate a minimal unified diff that implements the planner directive exactly. Edit only allowed files, do not create new files, and avoid unrelated refactors. Output ONLY unified diff text.

--- PLANNER DIRECTIVE ---
REQUIRES_CODE_CHANGE: YES
CONFIDENCE: MEDIUM
REASON: Issue requires code modifications in target files.
PLAN:
- Root cause: Fix default conn for BigQueryTableSensor



Make sure you have checked _all_ steps below.



### Jira



- [x] My PR addresses the following [Airflow-3966](https://issues.apache.org/jira/browse/AIRFLOW-3966) issues an

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def tokenize_example(ex: dict) -> dict:
    prompt = ex['input'].rstrip() + '\n'
    completion = ex['output']
    prompt_ids = tok(prompt, add_special_tokens=False)['input_ids']
    full_ids = tok(
        prompt + completion,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )['input_ids']
    labels = full_ids.copy()
    prompt_len = min(len(prompt_ids), len(full_ids))
    labels[:prompt_len] = [-100] * prompt_len
    return {
        'input_ids': full_ids,
        'attention_mask': [1] * len(full_ids),
        'labels': labels,
    }

train_tok = [tokenize_example(x) for x in tqdm(train_examples, desc='tokenize train')]
eval_tok = [tokenize_example(x) for x in tqdm(eval_examples, desc='tokenize eval')]

print('tokenized train/eval:', len(train_tok), len(eval_tok))


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenize eval: 100%|██████████| 158/158 [00:01<00:00, 122.63it/s]

tokenized train/eval: 1382 158


In [ ]:
from torch.utils.data import Dataset

class PatchDS(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        return self.rows[idx]

train_ds = PatchDS(train_tok)
eval_ds = PatchDS(eval_tok)

print('dataset sizes:', len(train_ds), len(eval_ds))


dataset sizes: 1382 158


In [ ]:
import os
from dataclasses import dataclass
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, EarlyStoppingCallback

os.makedirs(OUT_DIR, exist_ok=True)
torch.cuda.empty_cache()

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map='auto')
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

@dataclass
class CausalCollator:
    pad_token_id: int
    label_pad_token_id: int = -100
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        mx = max(len(f['input_ids']) for f in features)
        ii, am, lb = [], [], []
        for f in features:
            p = mx - len(f['input_ids'])
            ii.append(f['input_ids'] + [self.pad_token_id] * p)
            am.append(f['attention_mask'] + [0] * p)
            lb.append(f['labels'] + [self.label_pad_token_id] * p)
        return {
            'input_ids': torch.tensor(ii, dtype=torch.long),
            'attention_mask': torch.tensor(am, dtype=torch.long),
            'labels': torch.tensor(lb, dtype=torch.long),
        }

common_args = dict(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.08,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to='none',
    seed=SEED,
)

try:
    train_args = TrainingArguments(**common_args, group_by_length=True)
except TypeError:
    try:
        train_args = TrainingArguments(**common_args)
    except TypeError:
        alt = dict(common_args)
        if 'eval_strategy' in alt:
            alt['evaluation_strategy'] = alt.pop('eval_strategy')
        train_args = TrainingArguments(**alt)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=CausalCollator(tok.pad_token_id),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

train_result = trainer.train()
print(train_result)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


Epoch,Training Loss,Validation Loss
1,0.355527,0.256901
2,0.312900,0.251200


TrainOutput(global_step=346, training_loss=0.3163751932237879, metrics={'train_runtime': 2585.0168, 'train_samples_per_second': 1.069, 'train_steps_per_second': 0.134, 'total_flos': 2.224343408211026e+17, 'train_loss': 0.3163751932237879, 'epoch': 2.0})


In [ ]:
# Save immediate training artifacts to Drive
import shutil

LAST_ADAPTER = f'{OUT_DIR}/last_adapter'
trainer.save_model(LAST_ADAPTER)
tok.save_pretrained(LAST_ADAPTER)

if os.path.exists(f'{OUT_DIR}/trainer_state.json'):
    shutil.copy2(f'{OUT_DIR}/trainer_state.json', f'{EXPORT_DIR}/trainer_state.json')

with open(f'{EXPORT_DIR}/train_result_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(train_result.metrics, f, indent=2)

with open(f'{EXPORT_DIR}/train_log_history.json', 'w', encoding='utf-8') as f:
    json.dump(getattr(trainer.state, 'log_history', []), f, indent=2)

print('Saved:', LAST_ADAPTER)
print('Export dir:', EXPORT_DIR)


Saved: /content/drive/MyDrive/patcher_final_overnight_run/run/last_adapter
Export dir: /content/drive/MyDrive/patcher_final_overnight_run/exports


## Checkpoint Selection by Patch Quality

Scores each checkpoint with generation metrics on eval subset:
- valid unified diff
- touches only allowed files
- hunk header overlap with gold
- minimality (edit size ratio)


In [ ]:
import glob
from peft import PeftModel
from transformers import AutoModelForCausalLM

def parse_pred_files(diff_text: str) -> set:
    return set(re.findall(r'^\+\+\+ b/(.+)$', diff_text, flags=re.MULTILINE))

def hunk_headers(diff_text: str) -> set:
    return set(re.findall(r'^@@ .*? @@', diff_text, flags=re.MULTILINE))

def edit_size(diff_text: str) -> int:
    adds = sum(1 for x in diff_text.splitlines() if x.startswith('+') and not x.startswith('+++'))
    dels = sum(1 for x in diff_text.splitlines() if x.startswith('-') and not x.startswith('---'))
    return adds + dels

def is_valid_unified_diff(text: str) -> bool:
    return text.startswith('--- a/') and ('\n+++ b/' in text) and ('\n@@' in text)

def parse_unified_files(diff_text: str):
    lines = diff_text.splitlines()
    files = []
    i = 0
    while i < len(lines):
        if not lines[i].startswith('--- a/'):
            i += 1
            continue
        old_path = lines[i][6:]
        i += 1
        if i >= len(lines) or not lines[i].startswith('+++ b/'):
            break
        new_path = lines[i][6:]
        i += 1
        body = []
        hheaders = []
        while i < len(lines) and not lines[i].startswith('--- a/'):
            body.append(lines[i])
            if lines[i].startswith('@@ '):
                hheaders.append(lines[i])
            i += 1
        files.append({'old': old_path, 'new': new_path, 'hunks': hheaders, 'body_lines': body})
    return files

def parse_hunk_header_line(hline: str):
    m = re.match(r'^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@', hline)
    if not m:
        return None
    return {
        'old_start': int(m.group(1)),
        'old_count': int(m.group(2) or '1'),
        'new_start': int(m.group(3)),
        'new_count': int(m.group(4) or '1'),
    }

def apply_patch_text_to_source(src: str, file_diff_text: str):
    src_lines = src.splitlines()
    out_lines = []
    cursor = 0
    lines = file_diff_text.splitlines()
    i = 0
    while i < len(lines):
        if not lines[i].startswith('@@ '):
            i += 1
            continue
        h = parse_hunk_header_line(lines[i])
        if h is None:
            return False, src
        old_idx = h['old_start'] - 1
        if old_idx < cursor or old_idx > len(src_lines):
            return False, src
        out_lines.extend(src_lines[cursor:old_idx])
        src_ptr = old_idx
        i += 1
        while i < len(lines) and not lines[i].startswith('@@ '):
            raw = lines[i]
            if raw.startswith('\\'):
                i += 1
                continue
            if raw == '':
                return False, src
            tag, content = raw[0], raw[1:]
            if tag == ' ':
                if src_ptr >= len(src_lines) or src_lines[src_ptr] != content:
                    return False, src
                out_lines.append(src_lines[src_ptr])
                src_ptr += 1
            elif tag == '-':
                if src_ptr >= len(src_lines) or src_lines[src_ptr] != content:
                    return False, src
                src_ptr += 1
            elif tag == '+':
                out_lines.append(content)
            else:
                return False, src
            i += 1
        cursor = src_ptr
    out_lines.extend(src_lines[cursor:])
    patched = '\n'.join(out_lines)
    if src.endswith('\n'):
        patched += '\n'
    return True, patched

def git_show_file_at_sha(repo_dir: str, sha: str, file_path: str):
    import subprocess
    p = subprocess.run(
        ['git', '-C', repo_dir, 'show', f'{sha}:{file_path}'],
        capture_output=True,
        text=True,
    )
    if p.returncode != 0:
        return None
    return p.stdout

def apply_and_parse_pass(pred_text: str, ex_meta: dict):
    if not is_valid_unified_diff(pred_text):
        return False
    files = parse_unified_files(pred_text)
    if not files:
        return False
    base_sha = ex_meta.get('base_sha')
    if not base_sha:
        return False
    for f in files:
        fpath = f.get('old')
        if not fpath:
            return False
        src = git_show_file_at_sha(REPO_DIR, base_sha, fpath)
        if src is None:
            return False
        file_patch_text = '\n'.join(f.get('body_lines', []))
        ok, _ = apply_patch_text_to_source(src, file_patch_text)
        if not ok:
            return False
    return True

def load_eval_model_for_checkpoint(ckpt_path: str):
    # IMPORTANT: load a fresh base model per checkpoint to avoid adapter stacking.
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb,
        device_map='auto',
    )
    m = PeftModel.from_pretrained(base, ckpt_path)
    m.eval()
    return m

checkpoints = sorted(glob.glob(f'{OUT_DIR}/checkpoint-*'))
print('checkpoints:', checkpoints)

subset = eval_examples[:min(80, len(eval_examples))]
scores = []

for ck_idx, ck in enumerate(checkpoints):
    m = load_eval_model_for_checkpoint(ck)
    agg = {
        'valid': 0,
        'allowed': 0,
        'hunk_iou_sum': 0.0,
        'minimality_sum': 0.0,
        'exact_match': 0,
        'apply_parse_pass': 0,
    }

    for ex_idx, ex in enumerate(tqdm(subset, desc=f'eval {os.path.basename(ck)}', leave=False)):
        inp = tok(ex['input'], return_tensors='pt').to(m.device)
        with torch.no_grad():
            out = m.generate(
                **inp,
                max_new_tokens=PATCHER_MAX_NEW_TOKENS,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
                pad_token_id=tok.eos_token_id,
            )
        pred = tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        gold = ex['output'].strip()

        # Quick sanity preview to catch invalid generations early.
        if ck_idx == 0 and ex_idx < 2:
            print(f"\n[preview {os.path.basename(ck)} sample {ex_idx}]\n{pred[:500]}\n")

        agg['exact_match'] += int(pred == gold)

        valid = int(is_valid_unified_diff(pred))
        agg['valid'] += valid

        p_files = parse_pred_files(pred)
        allowed = set(ex['meta']['allowed_files'])
        agg['allowed'] += int(bool(p_files) and p_files.issubset(allowed))

        ph, gh = hunk_headers(pred), set(ex['meta']['gold_hunk_headers'])
        inter = len(ph & gh)
        union = max(1, len(ph | gh))
        agg['hunk_iou_sum'] += inter / union

        pe, ge = edit_size(pred), edit_size(gold)
        ratio = pe / max(1, ge)
        agg['minimality_sum'] += max(0.0, 1.0 - abs(1.0 - ratio))

        agg['apply_parse_pass'] += int(apply_and_parse_pass(pred, ex['meta']))

    n = max(1, len(subset))
    rec = {
        'checkpoint': ck,
        'valid_rate': agg['valid'] / n,
        'allowed_rate': agg['allowed'] / n,
        'hunk_iou': agg['hunk_iou_sum'] / n,
        'minimality': agg['minimality_sum'] / n,
        'exact_match_rate': agg['exact_match'] / n,
        'apply_parse_pass_rate': agg['apply_parse_pass'] / n,
    }
    # Weighted for correctness over format once valid/allowed are saturated.
    rec['score'] = (
        0.15 * rec['valid_rate']
        + 0.15 * rec['allowed_rate']
        + 0.30 * rec['hunk_iou']
        + 0.15 * rec['minimality']
        + 0.10 * rec['exact_match_rate']
        + 0.15 * rec['apply_parse_pass_rate']
    )
    scores.append(rec)

    del m
    torch.cuda.empty_cache()

scores = sorted(scores, key=lambda x: x['score'], reverse=True)
print(json.dumps(scores, indent=2))

BEST_CKPT = scores[0]['checkpoint'] if scores else None
print('BEST_CKPT:', BEST_CKPT)


checkpoints: ['/content/drive/MyDrive/patcher_final_overnight_run/run/checkpoint-173', '/content/drive/MyDrive/patcher_final_overnight_run/run/checkpoint-346']


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

eval checkpoint-173:   1%|▏         | 1/80 [01:17<1:42:27, 77.82s/it]


[preview checkpoint-173 sample 0]
--- a/airflow/utils/retries.py
+++ b/airflow/utils/retries.py
@@ -31,6 +31,7 @@
 
 MAX_DB_RETRIES = conf.getint('core', 'max_db_retries', fallback=3)
 
+SQLALCHEMY_EXCEPTION_TYPES = (OperationalError,)
 
 
 def run_with_db_retries(max_retries: int = MAX_DB_RETRIES, logger: Optional[logging.Logger] = None, **kwargs):
@@ -40,7 +41,7 @@ def run_with_db_retries(max_retries: int = MAX_DB_RETRIES, logger: Optional[loggi
         retry=tenacity.retry_if_exception_type(exception_types=OperationalError),



eval checkpoint-173:   2%|▎         | 2/80 [03:21<2:16:07, 104.71s/it]


[preview checkpoint-173 sample 1]
--- a/airflow/providers/cncf/kubernetes/operators/spark_kubernetes.py
+++ b/airflow/providers/cncf/kubernetes/operators/spark_kubernetes.py
@@ -69,7 +69,7 @@ class SparkKubernetesOperator(BaseOperator):
         self.api_group = api_group
         self.api_version = api_version
 
-    def execute(self, context):
+    def execute(self, context=None):
         self.log.info("Creating sparkApplication")
         hook = KubernetesHook(conn_id=self.kubernetes_conn_id)
         response = hook.create_



Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[
  {
    "checkpoint": "/content/drive/MyDrive/patcher_final_overnight_run/run/checkpoint-173",
    "valid_rate": 1.0,
    "allowed_rate": 0.875,
    "hunk_iou": 0.028306502525252526,
    "minimality": 0.26684567840077433,
    "exact_match_rate": 0.0,
    "apply_parse_pass_rate": 0.0,
    "score": 0.3297688025176919
  },
  {
    "checkpoint": "/content/drive/MyDrive/patcher_final_overnight_run/run/checkpoint-346",
    "valid_rate": 1.0,
    "allowed_rate": 0.875,
    "hunk_iou": 0.027594696969696974,
    "minimality": 0.22525950238382522,
    "exact_match_rate": 0.0,
    "apply_parse_pass_rate": 0.0,
    "score": 0.3233173344484829
  }
]
BEST_CKPT: /content/drive/MyDrive/patcher_final_overnight_run/run/checkpoint-173


In [ ]:
# Optional: N-best candidate reranking for inference/orchestrator use
# This demonstrates how to pick the best patch from multiple generations.

def rerank_score(pred: str, allowed_files: list[str], gold_hunks: list[str] | None = None):
    valid = 1.0 if is_valid_unified_diff(pred) else 0.0
    p_files = parse_pred_files(pred)
    allowed_ok = 1.0 if p_files and p_files.issubset(set(allowed_files)) else 0.0
    apply_ok = 0.0  # requires base_sha, so kept separate in full orchestrator
    if gold_hunks:
        ph = hunk_headers(pred)
        gh = set(gold_hunks)
        hunk_iou = len(ph & gh) / max(1, len(ph | gh))
    else:
        hunk_iou = 0.0
    # Inference rerank: prioritize validity + allowed-file compliance.
    score = 0.45 * valid + 0.35 * allowed_ok + 0.20 * hunk_iou
    return {
        'score': score,
        'valid': valid,
        'allowed_ok': allowed_ok,
        'hunk_iou': hunk_iou,
    }

def generate_n_and_rerank(model_for_infer, prompt: str, allowed_files: list[str], n: int = 3):
    inputs = tok(prompt, return_tensors='pt').to(model_for_infer.device)
    cands = []
    for i in range(n):
        with torch.no_grad():
            out = model_for_infer.generate(
                **inputs,
                max_new_tokens=PATCHER_MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.2,
                top_p=0.9,
                top_k=40,
                pad_token_id=tok.eos_token_id,
            )
        pred = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        info = rerank_score(pred, allowed_files)
        cands.append({'candidate_idx': i, 'pred': pred, **info})
    cands = sorted(cands, key=lambda x: x['score'], reverse=True)
    return cands

# Example usage (uncomment):
# infer_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map='auto')
# infer_model = PeftModel.from_pretrained(infer_base, BEST_CKPT).eval()
# sample = eval_examples[0]
# ranked = generate_n_and_rerank(infer_model, sample['input'], sample['meta']['allowed_files'], n=3)
# print('Best candidate score:', ranked[0]['score'])
# print(ranked[0]['pred'][:1200])

In [ ]:
import os, subprocess

REPO_DIR = "/content/airflow"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "https://github.com/apache/airflow.git", REPO_DIR],
        check=True
    )

# optional: quick sanity
subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--is-inside-work-tree"], check=True)
print("REPO_DIR ready:", REPO_DIR)

REPO_DIR ready: /content/airflow


In [ ]:
# Gold vs predicted apply baseline (same applier as checkpoint scoring)
# Run AFTER the checkpoint scoring cell (defines parse_unified_files, apply_and_parse_pass, REPO_DIR, tok, etc.).

import subprocess


def git_show_file_at_sha_debug(repo_dir: str, sha: str, file_path: str):
    p = subprocess.run(
        ['git', '-C', repo_dir, 'show', f'{sha}:{file_path}'],
        capture_output=True,
        text=True,
    )
    return p.returncode, p.stdout, (p.stderr or '').strip()


def diagnose_apply_unified_diff(diff_text: str, meta: dict):
    """Return ok + first failure reason for the notebook's line-based applier."""
    if not is_valid_unified_diff(diff_text):
        return False, 'invalid_unified_diff_shape'
    base_sha = meta.get('base_sha')
    if not base_sha:
        return False, 'missing_base_sha_in_meta'
    files = parse_unified_files(diff_text)
    if not files:
        return False, 'parse_unified_files_empty'
    for f in files:
        fpath = f.get('old')
        if not fpath:
            return False, 'missing_old_path'
        rc, src, err = git_show_file_at_sha_debug(REPO_DIR, base_sha, fpath)
        if rc != 0:
            return False, f'git_show_failed:{fpath}:{err[:200]}'
        file_patch_text = '\n'.join(f.get('body_lines', []))
        ok, _ = apply_patch_text_to_source(src, file_patch_text)
        if not ok:
            return False, f'apply_failed:{fpath}'
    return True, 'ok'


def load_best_for_smoke():
    if 'best_model' in globals() and globals()['best_model'] is not None:
        return globals()['best_model']
    if 'BEST_CKPT' not in globals() or not BEST_CKPT:
        raise RuntimeError('BEST_CKPT not set — run checkpoint scoring cell first.')
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb,
        device_map='auto',
    )
    m = PeftModel.from_pretrained(base, BEST_CKPT).eval()
    return m


GOLD_BASELINE_N = min(50, len(eval_examples))
PRED_SMOKE_K = min(5, len(eval_examples))
MAX_NEW_TOKENS_SMOKE = 1200  # raise if multi-file gold is long and preds truncate

gold_ok = 0
gold_fail = []
for i in range(GOLD_BASELINE_N):
    ex = eval_examples[i]
    gold = ex['output'].strip()
    ok, reason = diagnose_apply_unified_diff(gold, ex['meta'])
    gold_ok += int(ok)
    if not ok:
        gold_fail.append({'i': i, 'reason': reason})

print('Gold apply pass:', gold_ok, '/', GOLD_BASELINE_N)
if gold_fail:
    print('Gold failures (first 10):', json.dumps(gold_fail[:10], indent=2))

m_smoke = load_best_for_smoke()
pred_rows = []
for i in range(PRED_SMOKE_K):
    ex = eval_examples[i]
    inp = tok(ex['input'], return_tensors='pt').to(m_smoke.device)
    with torch.no_grad():
        out = m_smoke.generate(
            **inp,
            max_new_tokens=MAX_NEW_TOKENS_SMOKE,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tok.eos_token_id,
        )
    pred = tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    g_ok, g_reason = diagnose_apply_unified_diff(ex['output'].strip(), ex['meta'])
    p_ok, p_reason = diagnose_apply_unified_diff(pred, ex['meta'])
    pred_rows.append({
        'i': i,
        'gold_apply': g_ok,
        'gold_reason': g_reason,
        'pred_apply': p_ok,
        'pred_reason': p_reason,
        'pred_valid': is_valid_unified_diff(pred),
        'pred_len': len(pred),
    })

print('Pred smoke (same applier):')
print(json.dumps(pred_rows, indent=2))


NameError: name 'eval_examples' is not defined

In [ ]:
# Save best adapter + eval artifacts to Drive
BEST_ADAPTER = f'{OUT_DIR}/best_adapter'
if BEST_CKPT is not None:
    best_base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb,
        device_map='auto',
    )
    best_model = PeftModel.from_pretrained(best_base, BEST_CKPT)
    best_model.save_pretrained(BEST_ADAPTER)
    tok.save_pretrained(BEST_ADAPTER)
    del best_model
    del best_base
    torch.cuda.empty_cache()

with open(f'{EXPORT_DIR}/checkpoint_scores.json', 'w', encoding='utf-8') as f:
    json.dump(scores, f, indent=2)

with open(f'{EXPORT_DIR}/best_checkpoint_meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'best_checkpoint': BEST_CKPT,
        'best_adapter': BEST_ADAPTER,
        'score': scores[0]['score'] if scores else None,
        'model_name': MODEL_NAME,
        'max_seq_len': MAX_SEQ_LEN,
        'patcher_max_new_tokens': PATCHER_MAX_NEW_TOKENS,
        'num_epochs': NUM_EPOCHS,
        'lr': LR,
    }, f, indent=2)

print('Saved BEST_ADAPTER:', BEST_ADAPTER)
print('Saved exports in:', EXPORT_DIR)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Saved BEST_ADAPTER: /content/drive/MyDrive/patcher_final_overnight_run/run/best_adapter
Saved exports in: /content/drive/MyDrive/patcher_final_overnight_run/exports


In [ ]:
# Finalize overnight run: persist trainer state/logs and optionally kill runtime
import time
import signal

# Save trainer state if available
state_json = f'{OUT_DIR}/trainer_state.json'
if os.path.exists(state_json):
    with open(state_json, 'r', encoding='utf-8') as f:
        trainer_state = json.load(f)
else:
    trainer_state = {'note': 'trainer_state.json not found'}

with open(f'{EXPORT_DIR}/trainer_state_snapshot.json', 'w', encoding='utf-8') as f:
    json.dump(trainer_state, f, indent=2)

# Save training/eval log history
log_history = []
try:
    log_history = trainer.state.log_history
except Exception:
    log_history = []
with open(f'{EXPORT_DIR}/trainer_log_history.json', 'w', encoding='utf-8') as f:
    json.dump(log_history, f, indent=2)

print('Saved final snapshots to:', EXPORT_DIR)

# Toggle to True for unattended overnight shutdown after all saves.
KILL_RUNTIME_AFTER_SAVE = True
if KILL_RUNTIME_AFTER_SAVE:
    print('All artifacts saved. Shutting down runtime in 5 seconds...')
    time.sleep(5)
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print('Runtime kept alive (KILL_RUNTIME_AFTER_SAVE=False).')


Saved final snapshots to: /content/drive/MyDrive/patcher_final_overnight_run/exports
All artifacts saved. Shutting down runtime in 5 seconds...
